In [1]:
# =========================================================
# Quantile Spillover (Baseline Quantile-VAR Connectedness)
# - Rolling Quantile VAR(p)
# - Generalized FEVD based on quantile-regression coefficients
# - tau = 0.05, 0.50, 0.95
# =========================================================

import os
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from numpy.linalg import pinv
from statsmodels.regression.quantile_regression import QuantReg

# -----------------------------
# Config
# -----------------------------
DATA_PATH = "./spillover_data.csv"
OUT_DIR = "./output_quantile_spillover"

DATE_COL = "Date"
VARS = ["dlog_SOLVPN", "dlog_COPPER", "dlog_DXY", "d_UST10Y", "dlog_VIX"]

P = 1
H = 10
WINDOW = 200
TAUS = [0.05, 0.50, 0.95]

os.makedirs(OUT_DIR, exist_ok=True)

# -----------------------------
# Load
# -----------------------------
df = pd.read_csv(DATA_PATH)
df[DATE_COL] = pd.to_datetime(df[DATE_COL])
df = df.sort_values(DATE_COL).reset_index(drop=True)
df = df[[DATE_COL] + VARS].dropna().reset_index(drop=True)

# -----------------------------
# Helpers
# -----------------------------
def make_var_lagged(df_y: pd.DataFrame, p: int):
    X_list = []
    colnames = []
    for lag in range(1, p + 1):
        lagged = df_y.shift(lag)
        lagged.columns = [f"{c}_L{lag}" for c in df_y.columns]
        X_list.append(lagged)
        colnames.extend(lagged.columns.tolist())
    X = pd.concat(X_list, axis=1)
    out = pd.concat([df_y, X], axis=1).dropna().reset_index(drop=True)
    return out, colnames

def var_companion(A_list):
    p = len(A_list)
    k = A_list[0].shape[0]
    top = np.hstack(A_list)
    if p == 1:
        return top
    bottom = np.hstack([np.eye(k*(p-1)), np.zeros((k*(p-1), k))])
    return np.vstack([top, bottom])

def generalized_fevd(A_list, Sigma, H):
    k = Sigma.shape[0]
    p = len(A_list)
    F = var_companion(A_list)
    kp = k * p if p > 1 else k
    J = np.hstack([np.eye(k), np.zeros((k, kp-k))]) if p > 1 else np.eye(k)

    Phi = []
    Fpow = np.eye(kp)
    for h in range(H):
        Phi.append(J @ Fpow @ J.T)
        Fpow = Fpow @ F

    theta = np.zeros((k, k))
    sigdiag = np.diag(Sigma).copy()
    sigdiag[sigdiag <= 1e-12] = 1e-12

    for i in range(k):
        for j in range(k):
            e_i = np.zeros((k, 1)); e_i[i, 0] = 1
            e_j = np.zeros((k, 1)); e_j[j, 0] = 1
            num = 0.0
            den = 0.0
            for h in range(H):
                ph = Phi[h]
                num += float((e_i.T @ ph @ Sigma @ e_j) ** 2 / sigdiag[j])
                den += float(e_i.T @ ph @ Sigma @ ph.T @ e_i)
            theta[i, j] = num / max(den, 1e-12)

    rowsum = theta.sum(axis=1, keepdims=True)
    rowsum[rowsum <= 1e-12] = 1e-12
    return theta / rowsum

def spillover_from_fevd(theta):
    k = theta.shape[0]
    total = 100.0 * (theta.sum() - np.trace(theta)) / k
    to_ = 100.0 * (theta.sum(axis=0) - np.diag(theta))
    from_ = 100.0 * (theta.sum(axis=1) - np.diag(theta))
    net_ = to_ - from_
    return total, to_, from_, net_

# -----------------------------
# Prepare lagged data
# -----------------------------
base, lag_cols = make_var_lagged(df[VARS], P)
dates2 = df[DATE_COL].iloc[base.index].reset_index(drop=True)
base = base.reset_index(drop=True)

results = []
pairwise = []

for tau in TAUS:
    print(f"\nRunning tau={tau}")

    for end_idx in range(WINDOW, len(base) + 1):
        sub = base.iloc[end_idx - WINDOW:end_idx].copy()

        Y_sub = sub[VARS].values
        X_sub = sub[lag_cols].values
        X_sub = np.column_stack([np.ones(len(X_sub)), X_sub])

        k = len(VARS)
        A_list = [np.zeros((k, k)) for _ in range(P)]
        resid_mat = np.zeros((len(sub), k))

        for eq, var in enumerate(VARS):
            y = sub[var].values
            model = QuantReg(y, X_sub)
            fit = model.fit(q=tau, max_iter=5000)

            beta = fit.params
            resid = y - fit.predict(X_sub)
            resid_mat[:, eq] = resid

            coef_no_const = beta[1:]
            for lag in range(P):
                A_list[lag][eq, :] = coef_no_const[lag*k:(lag+1)*k]

        Sigma = np.cov(resid_mat.T)
        Sigma = (Sigma + Sigma.T) / 2
        d = np.diag(Sigma).copy()
        d[d < 1e-8] = 1e-8
        np.fill_diagonal(Sigma, d)

        theta = generalized_fevd(A_list, Sigma, H)
        total, to_, from_, net_ = spillover_from_fevd(theta)

        row = {
            "Date": dates2.iloc[end_idx - 1],
            "Tau": tau,
            "Total_Spillover": total
        }
        for i, v in enumerate(VARS):
            row[f"{v}_TO"] = to_[i]
            row[f"{v}_FROM"] = from_[i]
            row[f"{v}_NET"] = net_[i]

        results.append(row)

        for i, vi in enumerate(VARS):
            for j, vj in enumerate(VARS):
                pairwise.append({
                    "Date": dates2.iloc[end_idx - 1],
                    "Tau": tau,
                    "Response": vi,
                    "Shock": vj,
                    "FEVD": theta[i, j]
                })

res_df = pd.DataFrame(results)
pair_df = pd.DataFrame(pairwise)

# -----------------------------
# Save
# -----------------------------
res_df.to_csv(os.path.join(OUT_DIR, "quantile_connectedness.csv"), index=False)
pair_df.to_csv(os.path.join(OUT_DIR, "quantile_pairwise_fevd.csv"), index=False)

for tau in TAUS:
    avg_tau = (
        pair_df[pair_df["Tau"] == tau]
        .groupby(["Response", "Shock"])["FEVD"]
        .mean()
        .unstack()
        .reindex(index=VARS, columns=VARS)
    )
    avg_tau.to_csv(os.path.join(OUT_DIR, f"avg_fevd_tau_{str(tau).replace('.', '_')}.csv"))

meta = {
    "model": "Quantile-VAR Connectedness (baseline implementation)",
    "vars": VARS,
    "lag_order": P,
    "fevd_horizon": H,
    "window": WINDOW,
    "taus": TAUS
}
with open(os.path.join(OUT_DIR, "metadata.json"), "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2, default=str)

# -----------------------------
# Plot
# -----------------------------
plt.figure(figsize=(12, 5))
for tau in TAUS:
    tmp = res_df[res_df["Tau"] == tau].copy()
    plt.plot(pd.to_datetime(tmp["Date"]), tmp["Total_Spillover"], label=f"tau={tau}")
plt.title("Quantile Spillover: Total Connectedness")
plt.xlabel("Date")
plt.ylabel("Connectedness (%)")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "quantile_total_spillover.png"), dpi=300)
plt.show()

plt.figure(figsize=(12, 5))
for tau in TAUS:
    tmp = res_df[res_df["Tau"] == tau].copy()
    plt.plot(pd.to_datetime(tmp["Date"]), tmp["dlog_SOLVPN_NET"], label=f"SOLVPN tau={tau}")
plt.title("Quantile Spillover: SOLVPN NET")
plt.xlabel("Date")
plt.ylabel("NET")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "quantile_SOLVPN_net.png"), dpi=300)
plt.show()

print("Saved to:", OUT_DIR)
print(res_df.tail())

FileNotFoundError: [Errno 2] No such file or directory: './spillover_data.csv'